### 预先注册的动态工具
#### 根据 state 选择工具

In [21]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from collections.abc import Callable
from langchain.tools import tool
from typing import TypedDict, Annotated
from langchain.messages import AnyMessage
import operator

@tool
def public_search(query: str) -> str:
    """Search the web for the given query and return a summary of the results. All agent requests can use this tool."""
    return f"Search results for '{query}'"

@tool
def private_search(query: str) -> str:
    """Search the web for the given query and return a summary of the results. This tool can be used when the agent is authorized."""
    return f"Private search results for '{query}'"

@tool
def advanced_search(query: str) -> str:
    """Search the web for the given query and return a summary of the results. This tool is used for deep searching and can only be utilized when the agent is authorized and the number of messages exceeds 5."""
    return f"Advanced search results for '{query}'"

@wrap_model_call
def state_based_tools(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Filter tools based on conversation State."""
    # Read from State: check if user has authenticated
    state = request.state
    is_authenticated = state.get("authenticated", False)
    message_count = len(state["messages"])
    print(is_authenticated, message_count)

    # Only enable sensitive tools after authentication
    if not is_authenticated:
        tools = [t for t in request.tools if t.name.startswith("public_")]
        request = request.override(tools=tools)
    elif message_count < 6:
        # Limit tools early in conversation
        tools = [t for t in request.tools if t.name != "advanced_search"]
        request = request.override(tools=tools)

    return handler(request)

class AgentState(TypedDict):
    authenticated: bool
    messages: Annotated[list[AnyMessage], operator.add]

agent = create_agent(
    model="deepseek-chat",
    tools=[public_search, private_search, advanced_search],
    middleware=[state_based_tools],
    state_schema=AgentState,
)

In [22]:
from langchain.messages import HumanMessage

response1 = agent.invoke({
    'messages': [
        HumanMessage('How many search tools do you have available? Please list them.')
    ]
})

False 1


In [23]:
response1['messages'][-1].pretty_print()

================================== Ai Message ==================================

I have one search tool available:

**public_search** - This tool allows me to search the web for a given query and return a summary of the results. It's available for all agent requests to use.

The tool takes a query string as a parameter and returns search results from the web. This is the only search-related tool currently available in my toolkit.


In [24]:
response2 = agent.invoke({
    'messages': [
        HumanMessage('How many search tools do you have available? Please list them.')
    ],
    'authenticated': True  # Simulate user authentication
})

True 1


In [25]:
response2['messages'][-1].pretty_print()

================================== Ai Message ==================================

I have two search tools available:

1. **public_search** - This tool allows me to search the web for a given query and return a summary of the results. It can be used for general web searches.

2. **private_search** - This tool also allows me to search the web for a given query and return a summary of the results. The description notes that this tool can be used when the agent is authorized, suggesting it might have access to different or more specialized information sources compared to the public_search tool.

Both tools take a single required parameter: a "query" string that specifies what to search for.


In [26]:
from langchain.messages import SystemMessage, AIMessage

response3 = agent.invoke({
    'messages': [
        SystemMessage('You are an assistant that helps users search the web.'),
        HumanMessage('Hello.'),
        AIMessage('Hi there! How can I assist you today?'),
        HumanMessage('I want to ask you some questions.'),
        AIMessage('Sure! What would you like to know?'),
        HumanMessage('How many search tools do you have available? Please list them.')
    ],
    'authenticated': True  # Simulate user authentication
})

True 6


In [27]:
response3['messages'][-1].pretty_print()

================================== Ai Message ==================================

I have three search tools available:

1. **public_search** - For general web searches that return summary results
2. **private_search** - For web searches when I'm authorized to use it  
3. **advanced_search** - For deep web searching, which can only be used when I'm authorized and the number of messages exceeds 5

Each tool allows me to search the web for information based on queries you provide. Is there something specific you'd like me to search for?


#### 根据 Store 选择工具

In [28]:
from dataclasses import dataclass
from langgraph.store.memory import InMemoryStore

@dataclass
class Context:
    user_id: str

@wrap_model_call
def store_based_tools(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Filter tools based on Store preferences."""
    user_id = request.runtime.context.user_id

    # Read from Store: get user's enabled features
    store = request.runtime.store
    feature_flags = store.get(("features",), user_id)

    if feature_flags:
        enabled_features = feature_flags.value.get("enabled_tools", [])
        # Only include tools that are enabled for this user
        tools = [t for t in request.tools if t.name in enabled_features]
        request = request.override(tools=tools)

    return handler(request)

store = InMemoryStore()
store.put(("features",), "user123", {"enabled_tools": ["public_search", "advanced_search"]})
store.put(("features",), "user456", {"enabled_tools": ["public_search", "private_search", "advanced_search"]})

agent = create_agent(
    model="deepseek-chat",
    tools=[public_search, private_search, advanced_search],
    middleware=[store_based_tools],
    context_schema=Context,
    store=store,
)

In [29]:
response4 = agent.invoke(
    {'messages': [HumanMessage('How many search tools do you have available? Please list them.')]},
    context=Context(user_id='user123'),
)

In [30]:
response4['messages'][-1].pretty_print()

================================== Ai Message ==================================

I have two search tools available:

1. **public_search** - This tool allows me to search the web for a given query and return a summary of the results. It's available for all agent requests.

2. **advanced_search** - This tool also searches the web for a given query and returns a summary of the results. However, this tool is specifically designed for deep searching and can only be utilized when the agent is authorized and the number of messages in the conversation exceeds 5.

Both tools take a query parameter (string type) that specifies what to search for, and they return summarized search results.


In [31]:
response5 = agent.invoke(
    {'messages': [HumanMessage('How many search tools do you have available? Please list them.')]},
    context=Context(user_id='user456'),
)

In [32]:
response5['messages'][-1].pretty_print()

================================== Ai Message ==================================

I have three search tools available:

1. **public_search** - Search the web for the given query and return a summary of the results. All agent requests can use this tool.

2. **private_search** - Search the web for the given query and return a summary of the results. This tool can be used when the agent is authorized.

3. **advanced_search** - Search the web for the given query and return a summary of the results. This tool is used for deep searching and can only be utilized when the agent is authorized and the number of messages exceeds 5.

The main difference between them is their accessibility requirements:
- **public_search** is available to all agents
- **private_search** requires authorization 
- **advanced_search** requires both authorization and a minimum of 5 messages in the conversation


#### 根据 Context 选择工具

In [33]:
@wrap_model_call
def context_based_tools(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Filter tools based on Runtime Context permissions."""
    # Read from Runtime Context: get user role
    if request.runtime is None or request.runtime.context is None:
        # If no context provided, default to viewer (most restrictive)
        user_id = "viewer"
    else:
        user_id = request.runtime.context.user_id

    if user_id == "admin":
        # Admins get all tools
        pass
    elif user_id == "editor":
        # Editors can't delete
        tools = [t for t in request.tools if t.name != "advanced_search"]
        request = request.override(tools=tools)
    else:
        # Viewers get read-only tools
        tools = [t for t in request.tools if t.name.startswith("public_")]
        request = request.override(tools=tools)

    return handler(request)

agent = create_agent(
    model="deepseek-chat",
    tools=[public_search, private_search, advanced_search],
    middleware=[context_based_tools],
    context_schema=Context,
)

In [37]:
response6 = agent.invoke(
    {'messages': [HumanMessage('How many search tools do you have available? Please list them.')]},
    context=Context(user_id='admin'),
)

In [39]:
response6['messages'][-1].pretty_print()

================================== Ai Message ==================================

I have three search tools available:

1. **public_search** - Search the web for the given query and return a summary of the results. All agent requests can use this tool.

2. **private_search** - Search the web for the given query and return a summary of the results. This tool can be used when the agent is authorized.

3. **advanced_search** - Search the web for the given query and return a summary of the results. This tool is used for deep searching and can only be utilized when the agent is authorized and the number of messages exceeds 5.

Each tool has different access requirements and capabilities:
- **public_search**: Available to all agents
- **private_search**: Requires authorization
- **advanced_search**: Requires both authorization and a minimum of 5 messages in the conversation


In [40]:
response7 = agent.invoke(
    {'messages': [HumanMessage('How many search tools do you have available? Please list them.')]},
    context=Context(user_id='editor'),
)

In [41]:
response7['messages'][-1].pretty_print()

================================== Ai Message ==================================

I have two search tools available:

1. **public_search** - This tool allows me to search the web for a given query and return a summary of the results. It can be used for general search requests.

2. **private_search** - This tool also allows me to search the web for a given query and return a summary of the results. The key difference is that this tool can be used when I'm specifically authorized to do so (likely for more sensitive or specialized searches).

Both tools accept a single required parameter: "query" (a string containing the search terms), and both return summaries of search results. The main distinction appears to be in the authorization context in which each can be used.


In [42]:
response8 = agent.invoke(
    {'messages': [HumanMessage('How many search tools do you have available? Please list them.')]},
    context=Context(user_id='viewer'),
)

In [43]:
response8['messages'][-1].pretty_print()

================================== Ai Message ==================================

I have one search tool available:

**public_search** - This tool allows me to search the web for a given query and return a summary of the results. It's the primary tool I can use to gather current information from the internet when answering questions that require up-to-date or external data.

The tool takes a query string as input and returns search results in a summarized format. This is particularly useful for finding current events, recent developments, factual information, or any other data that might not be in my training knowledge base.

Is there something specific you'd like me to search for using this tool?


### 运行时注册工具

In [44]:
from langchain.agents.middleware import AgentMiddleware, ToolCallRequest

@tool
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    assert city != ''
    return f"It's always sunny in {city}."

# A tool that will be added dynamically at runtime
@tool
def calculate_tip(bill_amount: float, tip_percentage: float = 20.0) -> str:
    """Calculate the tip amount for a bill."""
    tip = bill_amount * (tip_percentage / 100)
    return f"Tip: ${tip:.2f}, Total: ${bill_amount + tip:.2f}"

class DynamicToolMiddleware(AgentMiddleware):
    """Middleware that registers and handles dynamic tools."""

    def wrap_model_call(self, request: ModelRequest, handler):
        # Add dynamic tool to the request
        # This could be loaded from an MCP server, database, etc.
        updated = request.override(tools=[*request.tools, calculate_tip])
        return handler(updated)

    def wrap_tool_call(self, request: ToolCallRequest, handler):
        # Handle execution of the dynamic tool
        if request.tool_call["name"] == "calculate_tip":
            return handler(request.override(tool=calculate_tip))
        return handler(request)

agent = create_agent(
    model="deepseek-chat",
    tools=[get_weather],  # Only static tools registered here
    middleware=[DynamicToolMiddleware()],
)

# The agent can now use both get_weather AND calculate_tip
result = agent.invoke({
    "messages": [{"role": "user", "content": "Calculate a 20% tip on $85"}]
})

In [46]:
from pprint import pprint

pprint(result)

{'messages': [HumanMessage(content='Calculate a 20% tip on $85', additional_kwargs={}, response_metadata={}, id='75c01560-3e3a-433f-833e-b48816aed07f'),
              AIMessage(content="I'll calculate the tip for you.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 377, 'total_tokens': 447, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 192}, 'prompt_cache_hit_tokens': 192, 'prompt_cache_miss_tokens': 185}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache', 'id': '4b5aede2-7a2e-4fb6-8294-bca8800ba633', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d42ee-f771-7ee1-b213-36df003abe2b-0', tool_calls=[{'name': 'calculate_tip', 'args': {'bill_amount': 85, 'tip_percentage': 20}, 'id': 'call_00_0hDLjtNjdf2P3iBUWTkFoJQT', 'type': 'tool_call'}], invalid_tool_calls=[], usage_met